# Vaidya — Notebook 1: ETL & Data Validation
**Period:** December 2025  
**Author:** Your Name  
**Assisted by:** Claude AI (Anthropic) — code, architecture, QA logic  
**Last updated:** March 2026  

## QA Findings Log

| # | Finding | Impact | Action Taken |
|---|---|---|---|
| QA-1 | File 1 has only 3 rows — Dec 9 spillover | Low | Kept as-is |
| QA-2 | 5 distinct domains found in page_location | Medium | Normalised to vaidya.in |
| QA-3 | 9.7% sessions have no attribution data | Medium | Marked as null/unknown |
| QA-4 | Essay/spam URLs present in traffic | Low | Identified, not filtered |
| QA-5 | add_to_cart_custom_event has ₹ symbol in price fields | Low | Cleaned during flattening |
| QA-6 | 'Whatspp' and 'Whatsapp' duplicate source values | Low | Standardised to 'whatsapp' |
| QA-7 | tax column 100% null in purchase params | Low | Column dropped |

## Key Numbers
- Total raw rows: 5,895,271
- Date range: Dec 1 — Dec 31 2025
- Total sessions: 465,462
- Attribution coverage: 90.3%

## Section 1 — Imports and Configuration
All libraries and constants defined in one place.

In [2]:
import pandas as pd
import numpy as np
import glob
import os
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# Folder containing parquet files
PARQUET_PATH = "../raw_files/events_dump"

# Funnel events we care about
FUNNEL_EVENTS = [
    'session_start',
    'view_item',
    'view_product_page_loaded',
    'add_to_cart',
    'add_to_cart_custom_event',
    'begin_checkout',
    'gokwik_checkout_initiated',
    'add_shipping_info',
    'add_payment_info',
    'purchase'
]

# Params we extract from session_start
ATTRIBUTION_KEYS = [
    'source', 'medium', 'campaign', 'term', 'gclid'
]

# Params we extract from purchase
COMMERCE_KEYS = [
    'transaction_id', 'value', 'currency',
    'payment_type', 'coupon', 'shipping', 'tax', 'discount'
]

# Params we extract from add_to_cart_custom_event
CART_KEYS = [
    'product_name', 'product_price',
    'cart_value', 'cart_count', 'product_id'
]

print("Configuration loaded.")
print(f"Funnel events: {len(FUNNEL_EVENTS)}")

Configuration loaded.
Funnel events: 10


## Section 2 — Load All 32 Parquet Files
We load each file individually, tag it with source filename,
and stack into one dataframe. This gives full traceability.

In [3]:
files = glob.glob(os.path.join(PARQUET_PATH, "*.parquet"))
print(f"Files found: {len(files)}")

dfs = []
for f in sorted(files):
    df_temp = pd.read_parquet(f)
    df_temp['source_file'] = os.path.basename(f)
    dfs.append(df_temp)
    print(f"{os.path.basename(f)} — {len(df_temp):,} rows")

df_raw = pd.concat(dfs, ignore_index=True)

print(f"\nTotal rows : {len(df_raw):,}")
print(f"Columns    : {df_raw.columns.tolist()}")
print(f"Date range : {df_raw['date_ist'].min()} to {df_raw['date_ist'].max()}")

Files found: 32
events_000000000000.parquet — 164,139 rows
events_000000000001.parquet — 3 rows
events_000000000002.parquet — 210,636 rows
events_000000000003.parquet — 210,914 rows
events_000000000004.parquet — 183,266 rows
events_000000000005.parquet — 168,707 rows
events_000000000006.parquet — 212,173 rows
events_000000000007.parquet — 193,379 rows
events_000000000008.parquet — 212,535 rows
events_000000000009.parquet — 230,960 rows
events_000000000010.parquet — 151,282 rows
events_000000000011.parquet — 191,782 rows
events_000000000012.parquet — 182,345 rows
events_000000000013.parquet — 191,970 rows
events_000000000014.parquet — 173,903 rows
events_000000000015.parquet — 170,398 rows
events_000000000016.parquet — 192,417 rows
events_000000000017.parquet — 199,699 rows
events_000000000018.parquet — 158,897 rows
events_000000000019.parquet — 148,419 rows
events_000000000020.parquet — 174,982 rows
events_000000000021.parquet — 192,399 rows
events_000000000022.parquet — 194,958 rows
e

## Checking the file with 3 rows.

events_000000000001.parquet contains only 3 rows versus the expected ~150,000-230,000 range. Investigating before proceeding.

In [4]:
df_suspect = df_raw[df_raw['source_file'] == 'events_000000000001.parquet']

In [5]:
df_suspect.head()

,user_id,session_id,event,event_ts,date_ist,time_ist,page_location,page_type,page_load_ts,event_params,device,source_file
164139,1236372546.1765304998,1765304998,first_visit,1765304999626992,2025-12-09,2025-12-09 23:59:59 UTC+0530,https://keralaayurveda.com/products/chyavanprash,products,1765305001867007,"[{'key': 'session_engaged', 'value': {'string_...",mobile,events_000000000001.parquet
164140,1236372546.1765304998,1765304998,session_start,1765304999626992,2025-12-09,2025-12-09 23:59:59 UTC+0530,https://keralaayurveda.com/products/chyavanprash,products,1765305001867007,"[{'key': 'gad_source', 'value': {'string_value...",mobile,events_000000000001.parquet
164141,1236372546.1765304998,1765304998,view_product_page_loaded,1765304999626992,2025-12-09,2025-12-09 23:59:59 UTC+0530,https://keralaayurveda.com/products/chyavanprash,products,1765305001867007,"[{'key': 'gad_source', 'value': {'string_value...",mobile,events_000000000001.parquet


## Section 3 — Drop Redundant Columns

- event_ts: Unix microsecond timestamp — replaced by time_ist
- page_load_ts: Identical to event_ts — redundant
- domain: Temporary investigation column

In [6]:
df_clean = df_raw.drop(columns=['event_ts', 'page_load_ts'])

print(f"Columns after drop: {df_clean.columns.tolist()}")
print(f"Shape: {df_clean.shape}")

Columns after drop: ['user_id', 'session_id', 'event', 'date_ist', 'time_ist', 'page_location', 'page_type', 'event_params', 'device', 'source_file']
Shape: (5895271, 10)


## Section 4 — Fix Timestamp

time_ist is a plain text string. We convert it to a proper
datetime object in IST timezone and rename to event_time.

In [7]:
df_clean['event_time'] = pd.to_datetime(
    df_clean['time_ist'],
    utc=True
).dt.tz_convert('Asia/Kolkata')

df_clean = df_clean.drop(columns=['time_ist'])

print(f"Sample timestamps:")
print(df_clean['event_time'].head(3).tolist())
print(f"\nNull count: {df_clean['event_time'].isnull().sum():,}")
print(f"Shape: {df_clean.shape}")

Sample timestamps:
[Timestamp('2025-12-20 03:02:33+0530', tz='Asia/Kolkata'), Timestamp('2025-12-20 03:02:33+0530', tz='Asia/Kolkata'), Timestamp('2025-12-20 03:02:33+0530', tz='Asia/Kolkata')]

Null count: 0
Shape: (5895271, 10)


## Section 5 — Filter to Funnel Events

We now filter to only the 10 funnel events relevant to our analysis.
Full dataset kept in df_clean. Filtered dataset saved as df_funnel.

In [8]:
df_funnel = df_clean[
    df_clean['event'].isin(FUNNEL_EVENTS)
].copy()

print(f"Full dataset rows  : {len(df_clean):,}")
print(f"Funnel event rows  : {len(df_funnel):,}")
print(f"Rows dropped       : {len(df_clean) - len(df_funnel):,}")
print(f"\nEvent distribution:")
print(df_funnel['event'].value_counts())

Full dataset rows  : 5,895,271
Funnel event rows  : 1,814,094
Rows dropped       : 4,081,177

Event distribution:
event
view_product_page_loaded     564583
view_item                    517765
session_start                465462
add_to_cart                   89312
add_to_cart_custom_event      74283
begin_checkout                28642
gokwik_checkout_initiated     28355
add_shipping_info             21877
add_payment_info              14943
purchase                       8872
Name: count, dtype: int64


## Section 6 — Flatten session_start (Attribution)

We extract attribution params from session_start events only.
These tell us where each session came from.
Result: one row per session with source, medium, campaign.

In [9]:
df_ss = df_funnel[df_funnel['event'] == 'session_start'].copy()

def flatten_attribution(params_array):
    result = {k: None for k in ATTRIBUTION_KEYS}
    if params_array is None:
        return result
    for param in params_array:
        key = param.get('key')
        if key not in ATTRIBUTION_KEYS:
            continue
        val = param.get('value', {})
        result[key] = (
            val.get('string_value') or
            val.get('int_value') or
            val.get('float_value') or
            val.get('double_value')
        )
    return result

attribution_flat = df_ss['event_params'].apply(flatten_attribution)
df_attribution = pd.DataFrame(attribution_flat.tolist())
df_attribution['session_id'] = df_ss['session_id'].values
df_attribution['event_time'] = df_ss['event_time'].values

print(f"Shape: {df_attribution.shape}")
print(f"\nSource distribution (top 10):")
print(df_attribution['source'].value_counts().head(10))
print(f"\nNull rates:")
for col in ATTRIBUTION_KEYS:
    pct = df_attribution[col].isnull().mean()*100
    print(f"  {col}: {pct:.1f}% null")

Shape: (465462, 7)

Source distribution (top 10):
source
google                348394
facebook               43098
Whatspp                 6831
youtube.com             5560
skendra                 3346
Limechat                2283
wishlink                2170
Whatsapp                1030
shopifybooster.pro       988
Linktree                 805
Name: count, dtype: int64

Null rates:
  source: 9.7% null
  medium: 9.8% null
  campaign: 9.8% null
  term: 36.6% null
  gclid: 33.0% null


##  Clean Attribution Data

Fix typo inconsistencies in source column.
'Whatspp' and 'Whatsapp' are the same channel.
Standardise to lowercase for consistency.

In [13]:
# Standardise source values
df_attribution['source'] = df_attribution['source'].str.lower().str.strip()
df_attribution['medium'] = df_attribution['medium'].str.lower().str.strip()

# Fix known typos
source_fixes = {
    'whatspp'  : 'whatsapp',
    'whatsapp' : 'whatsapp',
}

df_attribution['source'] = df_attribution['source'].replace(source_fixes)

# Verify fix
print("Source distribution after fix (top 10):")
print(df_attribution['source'].value_counts().head(10))

# Drop tax from commerce — 100% null
df_commerce = df_commerce.drop(columns=['tax'])
print(f"\nCommerce columns after dropping tax:")
print(df_commerce.columns.tolist())

Source distribution after fix (top 10):
source
google                348394
facebook               43098
whatsapp                7867
youtube.com             5560
skendra                 3346
limechat                2283
wishlink                2170
shopifybooster.pro       988
linktree                 805
bing                     532
Name: count, dtype: int64

Commerce columns after dropping tax:
['transaction_id', 'value', 'currency', 'payment_type', 'coupon', 'shipping', 'discount', 'session_id', 'event_time']


## Section 7 — Flatten purchase (Commerce Data)

We extract revenue params from purchase events only.
Critical for revenue, AOV, and deduplication analysis.

In [10]:
df_pur = df_funnel[df_funnel['event'] == 'purchase'].copy()

def flatten_commerce(params_array):
    result = {k: None for k in COMMERCE_KEYS}
    if params_array is None:
        return result
    for param in params_array:
        key = param.get('key')
        if key not in COMMERCE_KEYS:
            continue
        val = param.get('value', {})
        result[key] = (
            val.get('string_value') or
            val.get('int_value') or
            val.get('float_value') or
            val.get('double_value')
        )
    return result

commerce_flat = df_pur['event_params'].apply(flatten_commerce)
df_commerce = pd.DataFrame(commerce_flat.tolist())
df_commerce['session_id'] = df_pur['session_id'].values
df_commerce['event_time'] = df_pur['event_time'].values

print(f"Shape: {df_commerce.shape}")
print(f"\nSample:")
print(df_commerce[df_commerce['transaction_id'].notna()].head(3))
print(f"\nNull rates:")
for col in COMMERCE_KEYS:
    pct = df_commerce[col].isnull().mean()*100
    print(f"  {col}: {pct:.1f}% null")

Shape: (8872, 10)

Sample:
  transaction_id   value currency payment_type coupon  shipping   tax  \
0        #104722  1042.0      INR          upi     NA       NaN  None   
1        #104733  1365.0      INR          cod     NA      60.0  None   
2        #104699   675.0      INR          upi     NA       NaN  None   

   discount  session_id          event_time  
0     299.0  1766195809 2025-12-20 02:11:32  
1     299.0  1766199858 2025-12-20 03:05:51  
2     299.0  1766169738 2025-12-19 19:10:05  

Null rates:
  transaction_id: 0.0% null
  value: 0.0% null
  currency: 0.0% null
  payment_type: 0.1% null
  coupon: 0.1% null
  shipping: 37.7% null
  tax: 100.0% null
  discount: 68.0% null


## Section 8 — Flatten add_to_cart_custom_event (Cart Data)

We extract product and cart params from custom cart events.
Note: price fields contain ₹ symbol — cleaned to numeric here.

In [12]:
df_atc = df_funnel[
    df_funnel['event'] == 'add_to_cart_custom_event'
].copy()

def flatten_cart(params_array):
    result = {k: None for k in CART_KEYS}
    if params_array is None:
        return result
    for param in params_array:
        key = param.get('key')
        if key not in CART_KEYS:
            continue
        val = param.get('value', {})
        result[key] = (
            val.get('string_value') or
            val.get('int_value') or
            val.get('float_value') or
            val.get('double_value')
        )
    return result

cart_flat = df_atc['event_params'].apply(flatten_cart)
df_cart = pd.DataFrame(cart_flat.tolist())
df_cart['session_id'] = df_atc['session_id'].values
df_cart['event_time'] = df_atc['event_time'].values

# Clean ₹ symbol from price fields
for col in ['product_price', 'cart_value']:
    if col in df_cart.columns:
        df_cart[col] = (
            df_cart[col]
            .astype(str)
            .str.replace('₹', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df_cart[col] = pd.to_numeric(df_cart[col], errors='coerce')

print(f"Shape: {df_cart.shape}")
print(f"\nSample:")
print(df_cart[df_cart['product_name'].notna()].head(3))

Shape: (74283, 7)

Sample:
                       product_name  product_price  cart_value  cart_count  \
0  Neelibringadi Keram (Oil-200 Ml)          315.0       684.0         3.0   
1           Nalpamaradi Keram (Oil)          180.0       603.0         4.0   
2            Myaxyl Pain Relief Oil          437.0       437.0         1.0   

     product_id  session_id          event_time  
0  9.694440e+12  1766219022 2025-12-20 08:23:57  
1  9.694439e+12  1766189830 2025-12-20 00:50:06  
2  9.694440e+12  1766239263 2025-12-20 14:01:52  


In [15]:
df_funnel_clean = df_funnel.drop(columns=['event_params', 'source_file'])
print(f"Shape: {df_funnel_clean.shape}")
print(f"Columns: {df_funnel_clean.columns.tolist()}")

Shape: (1814094, 8)
Columns: ['user_id', 'session_id', 'event', 'date_ist', 'page_location', 'page_type', 'device', 'event_time']


In [19]:
DB_USER     = "rasaanj"
DB_PASSWORD = "3001"
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "vaidya"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [20]:
tables_to_check = [
    'funnel_events',
    'session_attribution', 
    'purchase_params',
    'cart_params'
]

with engine.connect() as conn:
    for table in tables_to_check:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).fetchone()[0]
        print(f"{table}: {count:,} rows")

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "funnel_events" does not exist
LINE 1: SELECT COUNT(*) FROM funnel_events
                             ^

[SQL: SELECT COUNT(*) FROM funnel_events]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [21]:
# Step 1 - verify connection first
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected:", result.fetchone()[0])

# Step 2 - load one table first as a test
print("\nLoading funnel_events...")
df_funnel_clean.to_sql(
    name      = 'funnel_events',
    con       = engine,
    if_exists = 'replace',
    index     = False,
    chunksize = 50000,
    method    = 'multi'
)
print("Done.")

# Step 3 - verify immediately
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM funnel_events")
    ).fetchone()[0]
    print(f"funnel_events: {count:,} rows")

Connected: PostgreSQL 15.16 (Homebrew) on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.6.3.2), 64-bit

Loading funnel_events...
Done.
funnel_events: 1,814,094 rows


In [22]:
remaining_tables = {
    'session_attribution' : df_attribution,
    'purchase_params'     : df_commerce,
    'cart_params'         : df_cart,
}

for table_name, df in remaining_tables.items():
    print(f"Loading {table_name}...")
    df.to_sql(
        name      = table_name,
        con       = engine,
        if_exists = 'replace',
        index     = False,
        chunksize = 50000,
        method    = 'multi'
    )
    with engine.connect() as conn:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).fetchone()[0]
        print(f"  ✓ {table_name}: {count:,} rows")

print("\nAll tables loaded. ETL complete.")

Loading session_attribution...
  ✓ session_attribution: 465,462 rows
Loading purchase_params...
  ✓ purchase_params: 8,872 rows
Loading cart_params...
  ✓ cart_params: 74,283 rows

All tables loaded. ETL complete.


## ETL Summary

| Table | Rows | Description |
|---|---|---|
| funnel_events | 1,814,094 | All 10 funnel events, clean |
| session_attribution | 465,462 | session_start attribution params |
| purchase_params | 8,872 | purchase commerce params |
| cart_params | 74,283 | add_to_cart_custom_event params |

ETL complete. All tables loaded to PostgreSQL (vaidya database).
Next: SQL QA checks and session_funnel build in Notebook 2.